In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import json
import matplotlib.pyplot as plt
import kagglehub
import os

c:\Users\mohak\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
player_dataset = kagglehub.dataset_download("swaptr/fifa-wc-2026-players")
teams_dataset = kagglehub.dataset_download("swaptr/fifa-wc-2026-teams")
matches_dataset= kagglehub.dataset_download("swaptr/fifa-wc-2026-matches")
print(os.listdir(player_dataset))
print(os.listdir(teams_dataset))
print(os.listdir(matches_dataset))

100%|██████████| 63.8k/63.8k [00:00<00:00, 144kB/s]

Extracting files...


100%|██████████| 7.78k/7.78k [00:00<00:00, 6.32MB/s]

Extracting files...


100%|██████████| 7.88k/7.88k [00:00<00:00, 13.9MB/s]

Extracting files...
['players.csv']
['teams.csv']
['matches.csv']


In [3]:
matches = pd.read_csv(f"{matches_dataset}/matches.csv")
players=pd.read_csv(f"{player_dataset}/players.csv")
teams=pd.read_csv(f"{teams_dataset}/teams.csv")

print(matches.columns.tolist())
print(players.columns.tolist())
print(teams.columns.tolist())


['round', 'gameweek', 'dayofweek', 'date', 'start_time', 'home_team', 'away_team', 'score', 'home_score', 'away_score', 'attendance', 'venue', 'referee', 'home_formation', 'away_formation', 'home_manager', 'away_manager', 'home_captain', 'away_captain', 'home_possession', 'away_possession', 'home_sot', 'away_sot', 'home_total_shots', 'away_total_shots', 'home_saves', 'away_saves', 'home_cards_yellow', 'away_cards_yellow', 'home_cards_red', 'away_cards_red', 'home_fouls', 'away_fouls', 'home_corners', 'away_corners', 'home_crosses', 'away_crosses', 'home_interceptions', 'away_interceptions', 'home_offsides', 'away_offsides', 'notes']
['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_targe

In [4]:
#Ref with most cards
ref_cards = matches.groupby('referee')[['home_cards_yellow','away_cards_yellow','home_cards_red','away_cards_red']].sum()
ref_cards['total_yellow'] = ref_cards['home_cards_yellow'] + ref_cards['away_cards_yellow']
ref_cards['total_red'] = ref_cards['home_cards_red'] + ref_cards['away_cards_red']
ref_cards['matches_officiated'] = matches.groupby('referee').size()
ref_cards['cards_per_match'] = (ref_cards['total_yellow'] + ref_cards['total_red']) / ref_cards['matches_officiated']

ref_cards = ref_cards.sort_values('cards_per_match', ascending=False).reset_index()
ref_cards.to_json("OutputData/GlobalWC/referee_cards.json",orient="records",indent=4,force_ascii=False)
print(ref_cards.head(10))

               referee  home_cards_yellow  away_cards_yellow  home_cards_red  \
0         Felix Zwayer                  6                  6               0   
1              Ma Ning                  1                  5               0   
2    François Letexier                  4                  9               0   
3       Michael Oliver                  7                 10               0   
4        João Pinheiro                  4                  6               0   
5        Slavko Vinčič                  3                 11               0   
6          Jalal Jayed                  4                  4               0   
7       Cristián Garay                  1                  1               0   
8  Alejandro Hernández                  1                  3               0   
9          Iván Barton                  8                  6               0   

   away_cards_red  total_yellow  total_red  matches_officiated  \
0               0            12          0           

In [5]:
#Top scorer and most minutes played

top_scorers = players.nlargest(10, 'goals')[['player','team','position','goals','goals_per90']]
top_minutes = players.nlargest(10, 'minutes')[['player','team','position','minutes','games','minutes_per_game']]

top_scorers.to_json("OutputData/GlobalWC/top_scorer.json",orient="records",indent=4,force_ascii=False)
top_minutes.to_json("OutputData/GlobalWC/top_minutes.json",orient="records",indent=4,force_ascii=False)

print(top_scorers.head(10))
print(top_minutes.head(10))

               player       team position  goals  goals_per90
477     Kylian Mbappé     France       FW   10.0         1.29
41       Lionel Messi  Argentina       FW    8.0         0.97
458   Jude Bellingham    England       MF    7.0         1.03
811    Erling Haaland     Norway       FW    7.0         1.35
451        Harry Kane    England       FW    6.0         0.83
488   Ousmane Dembélé     France       MF    6.0         0.91
1055  Mikel Oyarzabal      Spain       FW    5.0         0.75
179   Vinicius Júnior     Brazil       FW    4.0         0.82
717   Julián Quiñones     Mexico    FW,MF    4.0         0.88
1000     Ismaila Sarr    Senegal    MF,FW    4.0         0.99
                   player       team position  minutes  games  \
28      Emiliano Martínez  Argentina       GK    810.0      8   
26    Alexis Mac Allister  Argentina       MF    750.0      8   
1050       Marc Cucurella      Spain       DF    750.0      8   
1057          Pau Cubarsí      Spain       DF    750.0    

In [6]:
#Goals per venue
matches['total_goals'] = matches['home_score'] + matches['away_score']

goals_per_venue = matches.groupby('venue')['total_goals'].agg(['sum','mean','count']).reset_index()
goals_per_venue.columns = ['venue','total_goals','avg_goals_per_match','matches_played']
goals_per_venue = goals_per_venue.sort_values('total_goals', ascending=False)

goals_per_venue.to_json("OutputData/GlobalWC/goals_per_venue.json",orient="records",indent=4,force_ascii=False)

print(goals_per_venue.head(10))

                                           venue  total_goals  \
13                Mercedes-Benz Stadium, Atlanta         27.0   
9               Hard Rock Stadium, Miami Gardens         27.0   
15                      Reliant Stadium, Houston         27.0   
16                       SoFi Stadium, Inglewood         26.0   
0                        AT&T Stadium, Arlington         24.0   
1                    BC Place Stadium, Vancouver         23.0   
14              MetLife Stadium, East Rutherford         23.0   
12                          Lumen Field, Seattle         20.0   
7   GEHA Field at Arrowhead Stadium, Kansas City         18.0   
6                      Estadio Banorte, Coyoacán         16.0   

    avg_goals_per_match  matches_played  
13             3.375000               8  
9              3.857143               7  
15             3.857143               7  
16             3.250000               8  
0              3.000000               8  
1              3.833333         

In [7]:
# Most attended venue

matches['attendance'] = (
    matches['attendance']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
matches['attendance'] = pd.to_numeric(matches['attendance'], errors='coerce')
attendees = (matches.groupby("venue")['attendance'].sum().reset_index().sort_values("attendance",ascending=False))

attendees.to_json("OutputData/GlobalWC/attendees.json",orient="records",indent=4,force_ascii=False)
print(attendees.head(10))


                                           venue  attendance
14              MetLife Stadium, East Rutherford      645186
0                        AT&T Stadium, Arlington      631843
16                       SoFi Stadium, Inglewood      561656
13                Mercedes-Benz Stadium, Atlanta      544516
15                      Reliant Stadium, Houston      480184
9               Hard Rock Stadium, Miami Gardens      449157
8                   Gillette Stadium, Foxborough      447283
7   GEHA Field at Arrowhead Stadium, Kansas City      413169
10                   Levi's Stadium, Santa Clara      411345
11         Lincoln Financial Field, Philadelphia      409894


In [8]:
# club diversity

club_diversity = players.groupby('team')['club'].nunique().sort_values(ascending=False).reset_index()
club_diversity.columns = ['team','unique_clubs_represented']

club_diversity.to_json("OutputData/GlobalWC/club_diversity.json",orient="records",indent=4,force_ascii=False)

print(club_diversity.head(10))


             team  unique_clubs_represented
0   United States                        22
1     Switzerland                        22
2   Côte d'Ivoire                        21
3          Norway                        21
4           Japan                        21
5          Sweden                        21
6        Paraguay                        20
7         Morocco                        20
8  Korea Republic                        19
9       Argentina                        19


In [9]:
#club player counts

club_counts = players['club'].value_counts().head(15).reset_index()
club_counts.columns = ['club', 'player_count']

club_counts.to_json("OutputData/GlobalWC/club_counts.json",orient="records",indent=4,force_ascii=False)

print(club_counts.head(10))

              club  player_count
0  Manchester City            18
1    Bayern Munich            17
2          Arsenal            14
3              PSG            14
4        Barcelona            13
5   Crystal Palace            12
6      Galatasaray            12
7         Al-Hilal            12
8         Dortmund            11
9        Liverpool            10


In [10]:
#best formation

matches['home_won'] = matches['home_score'] > matches['away_score']
home_f = matches[['home_formation','home_won']].rename(columns={'home_formation':'formation','home_won':'won'})
away_f = matches[['away_formation']].copy()
away_f['won'] = ~matches['home_won']
away_f = away_f.rename(columns={'away_formation':'formation'})
formation_wins = pd.concat([home_f, away_f]).groupby('formation')['won'].sum().reset_index()
formation_wins.columns = ['formation', 'total_wins']
formation_wins = formation_wins.sort_values('total_wins', ascending=False)

formation_wins.to_json("OutputData/GlobalWC/formation_wins.json",orient="records",indent=4,force_ascii=False)
print(formation_wins.head(10))


   formation  total_wins
7    4-2-3-1          41
5    4-1-4-1          29
6    4-2-2-2          18
2      3-4-3           5
11     5-4-1           5
4    4-1-3-2           3
1    3-4-1-2           1
3      3-5-2           1
10     5-3-2           1
0    3-1-4-2           0


In [11]:
# cards by team

teams['cards_yellow'] = pd.to_numeric(teams['cards_yellow'], errors='coerce')
teams['cards_red'] = pd.to_numeric(teams['cards_red'], errors='coerce')
teams['total_cards'] = teams['cards_red'] + teams['cards_yellow']
cards_by_team = teams[['team','cards_yellow','cards_red','total_cards']].sort_values('total_cards', ascending=False)

cards_by_team.to_json("OutputData/GlobalWC/cards_by_team.json",orient="records",indent=4,force_ascii=False)

print(cards_by_team)

              team  cards_yellow  cards_red  total_cards
1        Argentina            15          1           16
16           Egypt            12          0           12
8           Canada            11          0           11
33        Paraguay             9          1           10
17         England             8          1            9
42     Switzerland             8          1            9
15         Ecuador             8          1            9
5      Bosnia–Herz             7          1            8
6           Brazil             8          0            8
9         Colombia             8          0            8
45   United States             7          1            8
34        Portugal             7          0            7
28         Morocco             7          0            7
21           Haiti             7          0            7
12         Curaçao             7          0            7
4          Belgium             6          1            7
39    South Africa             

C:\Users\mohak\AppData\Local\Temp\ipykernel_9604\313024180.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  teams['total_cards'] = teams['cards_red'] + teams['cards_yellow']


In [12]:
#Most goals per team

teams['goals'] = pd.to_numeric(teams['goals'], errors='coerce')
top_scoring_teams = teams.nlargest(10, 'goals')[['team','goals','goals_per90']]

top_scoring_teams.to_json("OutputData/GlobalWC/top_scoring_teams.json",orient="records",indent=4,force_ascii=False)

print(top_scoring_teams)

         team  goals  goals_per90
17    England     20         2.40
18     France     20         2.50
1   Argentina     18         2.00
4     Belgium     13         2.05
40      Spain     13         1.56
31     Norway     12         1.89
19    Germany     11         2.54
6      Brazil     10         2.00
27     Mexico     10         2.00
28    Morocco     10         1.58


In [13]:
# Goalkeeper with most shots saved
gk = players[players['position'] == 'GK'][['player','team','gk_saves']]
gk_rank = gk.sort_values('gk_saves', ascending=False)

gk_rank.to_json("OutputData/GlobalWC/gk_rank.json",orient="records",indent=4,force_ascii=False)

print(gk_rank.head(10))

                 player          team  gk_saves
881        Orlando Gill      Paraguay      23.0
28    Emiliano Martínez     Argentina      20.0
1103       Gregor Kobel   Switzerland      20.0
316           Eloy Room       Curaçao      20.0
887         Diogo Costa      Portugal      19.0
204             Vozinha    Cabo Verde      18.0
754     Bart Verbruggen   Netherlands      16.0
951   Mohammed Al-Owais  Saudi Arabia      16.0
831        Ørjan Nyland        Norway      15.0
457     Jordan Pickford       England      15.0
